# Advanced Indexing, Ufuncs and Sorting

## Quick notes

### What this covers
Index-based tools beyond basic slicing (`take`, `put`, `take_along_axis`, `ix_`, `ravel_multi_index`), ufunc methods (`reduce`, `accumulate`, `outer`, `at`, `reduceat`), why `np.vectorize` is not fast, `einsum`, lookup and counting tools (`searchsorted`, `digitize`, `unique`, `bincount`, `histogram`), set operations, sorting (`kind`, `partition`, `lexsort`), nan-aware functions, cumulative ops, `diff`/`gradient`/`convolve`/`correlate`, and rounding/clipping variants.

### Key points
- `np.take(a, idx, axis, mode)` = fancy indexing with an axis and `mode='clip'|'wrap'`; `np.put` writes into the **flattened** array.
- `take_along_axis` / `put_along_axis` apply per-row/column index arrays (the partner of `argsort`, `argmax(keepdims=True)`).
- `a[np.ix_(rows, cols)]` selects a sub-grid; `a[rows, cols]` selects paired points. `ravel_multi_index` / `unravel_index` convert between 2-D and flat positions.
- Every binary ufunc has `.reduce`, `.accumulate`, `.outer`, `.at`, `.reduceat`. `np.add.reduce` is `sum`, `np.maximum.accumulate` is a running max.
- `np.add.at(a, idx, v)` handles repeated indexes; `a[idx] += v` does not.
- `np.vectorize` and `np.frompyfunc` are syntax helpers - still one Python call per element. `apply_along_axis` is also a loop.
- `einsum`: letters name axes; letters missing after `->` are summed. Covers dot, outer, trace, transpose, matmul, batched matmul.
- `searchsorted` = binary search (insertion point); `digitize` = bin index; `bincount` = fast integer counts (with `weights=` for group sums); `unique(..., return_index/inverse/counts)`.
- `histogram` bins are half-open except the last; `histogram2d` rows are X bins.
- Set ops (`union1d`, `intersect1d`, `setdiff1d`, `setxor1d`) return sorted unique values; `np.isin` keeps shape and order.
- Sorting: `np.sort` copies, `a.sort()` is in place; `kind='stable'` keeps ties in order; `partition`/`argpartition` give O(n) top-k; `lexsort` sorts by many keys (last key is primary).
- `nansum`, `nanmean`, `nanmax`, ... skip nans. `cumsum`, `cumprod`, `diff`, `gradient`, `convolve`, `correlate` are the signal/series basics.
- `clip`, `round`/`rint` (half to even), `floor`, `ceil`, `trunc`, `floor_divide`, `mod` vs `fmod`.

### Gotchas (the part worth revising twice)
- `a[[0,0,1]] += 1` counts index 0 only once; use `np.add.at` or `np.bincount`.
- `searchsorted` and `digitize` need sorted bins; unsorted input gives silent garbage.
- `scores[argsort_result]` indexes rows, not within-row elements - use `take_along_axis`.
- `argmax` on 2-D returns a flat index; `np.unravel_index` gives (row, col).
- `np.vectorize` guesses output dtype from the first result (can truncate floats to ints) and fails on empty input without `otypes=`.
- `np.sort` default is not stable; multi-pass sorting requires `kind='stable'`. NaNs sort last.
- `lexsort` keys are reversed: the last key is the main one. Negate numbers for descending.
- `bincount` allocates `max+1` slots - huge labels waste memory; negatives raise `ValueError`.
- `np.round(2.5) == 2.0` (half to even); `astype(int)` truncates toward zero, not floor.
- `np.convolve` flips the kernel; `np.correlate` does not. `'same'` mode zero-pads the edges.
- `np.max` and `np.mean` propagate nan; `nanargmax` raises on an all-nan slice.
- `cumprod` overflows silently on fixed-size ints.

### When it matters
Group-by style work without pandas (`bincount` with weights, `add.at`), top-k selection, lookups of "which bucket / nearest value", per-row ranking, deduplication, and writing compact tensor code with `einsum`. Knowing which tools are really loops (`vectorize`, `apply_along_axis`) tells you where speed will not come from.

### One-line recap example
```python
import numpy as np
labels, vals = np.array([0, 2, 2, 1]), np.array([1.0, 2.0, 3.0, 4.0])
print(np.bincount(labels, weights=vals))   # group sums: [1. 4. 5.]
```

In [1]:
# 01. take and put
# 02. take_along_axis and put_along_axis
# 03. np.ix_ and ravel_multi_index
# 04. ufunc Methods: reduce, accumulate, outer
# 05. ufunc Methods: at and reduceat
# 06. frompyfunc and np.vectorize (and Why They Are Slow)
# 07. apply_along_axis and apply_over_axes
# 08. einsum Step by Step
# 09. searchsorted
# 10. digitize
# 11. np.unique with return_*
# 12. bincount
# 13. histogram and histogram2d
# 14. Set Operations
# 15. Sorting and Sort Kinds
# 16. partition and argpartition
# 17. lexsort
# 18. NaN-Aware Functions
# 19. Cumulative Operations
# 20. diff, gradient, convolve, correlate
# 21. clip, round, floor Variants
# 22. Cross-References
#
# Related: 02 (basic/fancy indexing, boolean masks), 05 (stats), 08 (vectorization),
# 10 (structured/masked arrays, select), 11 (views vs copies).

import warnings

import numpy as np

## 01. take and put

- np.take(a, indices, axis) = fancy indexing with an explicit axis and mode option.
- mode='clip' / 'wrap' handle out-of-range indexes instead of raising IndexError.
- np.put(a, indices, values) writes into the FLATTENED array in place.
- np.putmask / np.place write wherever a mask is True.

In [2]:
a = np.array([[10, 11, 12], [20, 21, 22], [30, 31, 32]])
print("a =\n", a)
print("np.take(a, [0, 2], axis=0) =\n", np.take(a, [0, 2], axis=0))
print("np.take(a, [0, 2], axis=1) =\n", np.take(a, [0, 2], axis=1))
print("np.take(a, [1, 10]) flat, mode='wrap':", np.take(a, [1, 10], mode="wrap"), "(10 wraps to 10 % 9 = 1)")
print("np.take(a, [1, 99], mode='clip'):", np.take(a, [1, 99], mode="clip"), "(99 -> last element)")
try:
    np.take(a, [99])
except IndexError as e:
    print("mode='raise' (default) ->", type(e).__name__)
p = np.arange(6)
np.put(p, [0, 5], [-1, -5])
print("np.put(p, [0,5], [-1,-5]):", p)
np.put(p, [1, 2, 3], [100])
print("values repeat if too few:", p)
np.putmask(p, p > 50, 0)
print("putmask(p > 50 -> 0):", p)
print("# Gotcha: np.put indexes the FLATTENED array, even for 2-D input.")
print("#   Plain fancy indexing a[idx] is just as good; take mostly adds axis= and mode=.")
print()

a =
 [[10 11 12]
 [20 21 22]
 [30 31 32]]
np.take(a, [0, 2], axis=0) =
 [[10 11 12]
 [30 31 32]]
np.take(a, [0, 2], axis=1) =
 [[10 12]
 [20 22]
 [30 32]]
np.take(a, [1, 10]) flat, mode='wrap': [11 11] (10 wraps to 10 % 9 = 1)
np.take(a, [1, 99], mode='clip'): [11 32] (99 -> last element)
mode='raise' (default) -> IndexError
np.put(p, [0,5], [-1,-5]): [-1  1  2  3  4 -5]
values repeat if too few: [ -1 100 100 100   4  -5]
putmask(p > 50 -> 0): [-1  0  0  0  4 -5]
# Gotcha: np.put indexes the FLATTENED array, even for 2-D input.
#   Plain fancy indexing a[idx] is just as good; take mostly adds axis= and mode=.



## 02. take_along_axis and put_along_axis

- Picks one (or several) element per row/column using an index array of matching shape.
- Typical use: apply the result of argsort / argmax per row.
- Index array must have the same ndim as the data; use keepdims=True.

In [3]:
scores = np.array([[0.2, 0.9, 0.4],
                   [0.7, 0.1, 0.6]])
order = np.argsort(scores, axis=1)
print("argsort per row:\n", order)
print("sorted rows via take_along_axis:\n", np.take_along_axis(scores, order, axis=1))
best = np.argmax(scores, axis=1, keepdims=True)
print("argmax(keepdims=True) shape:", best.shape)
print("best value per row:", np.take_along_axis(scores, best, axis=1).ravel())
top2 = order[:, ::-1][:, :2]
print("top-2 per row:\n", np.take_along_axis(scores, top2, axis=1))
z = np.zeros((2, 3), dtype=int)
np.put_along_axis(z, best, 1, axis=1)
print("one-hot per row via put_along_axis:\n", z)
print("# Gotcha: scores[order] would index ROWS, not elements within each row. For per-row")
print("#   reordering use take_along_axis (or scores[np.arange(n)[:, None], order]).")
print()

argsort per row:
 [[0 2 1]
 [1 2 0]]
sorted rows via take_along_axis:
 [[0.2 0.4 0.9]
 [0.1 0.6 0.7]]
argmax(keepdims=True) shape: (2, 1)
best value per row: [0.9 0.7]
top-2 per row:
 [[0.9 0.4]
 [0.7 0.6]]
one-hot per row via put_along_axis:
 [[0 1 0]
 [1 0 0]]
# Gotcha: scores[order] would index ROWS, not elements within each row. For per-row
#   reordering use take_along_axis (or scores[np.arange(n)[:, None], order]).



## 03. np.ix_ and ravel_multi_index

- np.ix_(rows, cols) builds an "open mesh" so a[np.ix_(r, c)] selects a SUB-GRID.
- Without ix_, a[[0,2],[0,2]] pairs indexes -> just 2 points (the diagonal).
- ravel_multi_index converts (row, col) pairs to flat positions; unravel_index goes back.

In [4]:
print("a[[0, 2], [0, 2]] (paired points) =", a[[0, 2], [0, 2]])
print("a[np.ix_([0, 2], [0, 2])] (sub-grid) =\n", a[np.ix_([0, 2], [0, 2])])
print("np.ix_ with a boolean mask:", a[np.ix_(np.array([True, False, True]), [1])].ravel())
flat = np.ravel_multi_index(([0, 1, 2], [2, 1, 0]), a.shape)
print("ravel_multi_index of (0,2),(1,1),(2,0):", flat, "-> values", a.ravel()[flat])
print("unravel_index(5, (3,3)) =", np.unravel_index(5, (3, 3)))
print("unravel_index of argmax:", np.unravel_index(np.argmax(a), a.shape), "(2-D position of the max)")
print("ravel_multi_index mode='clip':", np.ravel_multi_index(([5], [5]), (3, 3), mode="clip"))
print("# Gotcha: argmax on a 2-D array returns a FLAT index. Use unravel_index for (row, col).")
print()

a[[0, 2], [0, 2]] (paired points) = [10 32]
a[np.ix_([0, 2], [0, 2])] (sub-grid) =
 [[10 12]
 [30 32]]
np.ix_ with a boolean mask: [11 31]
ravel_multi_index of (0,2),(1,1),(2,0): [2 4 6] -> values [12 21 30]
unravel_index(5, (3,3)) = (1, 2)
unravel_index of argmax: (2, 2) (2-D position of the max)
ravel_multi_index mode='clip': [8]
# Gotcha: argmax on a 2-D array returns a FLAT index. Use unravel_index for (row, col).



## 04. ufunc Methods: reduce, accumulate, outer

- Every binary ufunc (add, multiply, maximum, logical_and, ...) has methods:
    reduce      fold along an axis -> one value per remaining position
    accumulate  running fold (keeps the intermediate results)
    outer       apply to all pairs (a_i op b_j)
- np.add.reduce(x) is what x.sum() calls.

In [5]:
x = np.array([1, 2, 3, 4])
print("np.add.reduce(x) =", np.add.reduce(x), ", np.multiply.reduce(x) =", np.multiply.reduce(x))
print("np.add.accumulate(x) =", np.add.accumulate(x), "(same as np.cumsum)")
print("np.maximum.accumulate([3,1,4,1,5,2]) =", np.maximum.accumulate([3, 1, 4, 1, 5, 2]), "(running max)")
print("np.logical_and.reduce([True, True, False]) =", np.logical_and.reduce([True, True, False]))
m = np.arange(6).reshape(2, 3)
print("np.add.reduce(m, axis=0) =", np.add.reduce(m, axis=0), ", axis=1 =", np.add.reduce(m, axis=1))
print("np.add.reduce(m, axis=None) =", np.add.reduce(m, axis=None))
print("np.add.reduce(m, axis=1, keepdims=True) shape:", np.add.reduce(m, axis=1, keepdims=True).shape)
print("np.add.reduce(x, initial=100) =", np.add.reduce(x, initial=100))
print("np.multiply.outer([1,2,3], [10,20]) =\n", np.multiply.outer([1, 2, 3], [10, 20]))
print("np.subtract.outer([5,6], [1,2,3]) =\n", np.subtract.outer([5, 6], [1, 2, 3]))
print("np.equal.outer(x, [2, 4]) =\n", np.equal.outer(x, [2, 4]))
print("np.power.outer([1,2,3], [1,2,3]) (a table of powers) =\n", np.power.outer([1, 2, 3], [1, 2, 3]))
print("# Gotcha: reduce on an EMPTY array needs an identity: np.add.reduce([]) is 0.0 but")
print("#   np.maximum.reduce([]) raises ValueError. Pass initial= for ufuncs without one.")
print()

np.add.reduce(x) = 10 , np.multiply.reduce(x) = 24
np.add.accumulate(x) = [ 1  3  6 10] (same as np.cumsum)
np.maximum.accumulate([3,1,4,1,5,2]) = [3 3 4 4 5 5] (running max)
np.logical_and.reduce([True, True, False]) = False
np.add.reduce(m, axis=0) = [3 5 7] , axis=1 = [ 3 12]
np.add.reduce(m, axis=None) = 15
np.add.reduce(m, axis=1, keepdims=True) shape: (2, 1)
np.add.reduce(x, initial=100) = 110
np.multiply.outer([1,2,3], [10,20]) =
 [[10 20]
 [20 40]
 [30 60]]
np.subtract.outer([5,6], [1,2,3]) =
 [[4 3 2]
 [5 4 3]]
np.equal.outer(x, [2, 4]) =
 [[False False]
 [ True False]
 [False False]
 [False  True]]
np.power.outer([1,2,3], [1,2,3]) (a table of powers) =
 [[ 1  1  1]
 [ 2  4  8]
 [ 3  9 27]]
# Gotcha: reduce on an EMPTY array needs an identity: np.add.reduce([]) is 0.0 but
#   np.maximum.reduce([]) raises ValueError. Pass initial= for ufuncs without one.



## 05. ufunc Methods: at and reduceat

- ufunc.at(a, idx, vals): UNBUFFERED in-place op. Repeated indexes all count.
- Plain a[idx] += 1 with repeated indexes applies only once per position.
- reduceat(a, starts): reduce over slices [starts[i] : starts[i+1]] in one call.

In [6]:
c1 = np.zeros(4, dtype=int)
c1[[0, 0, 1]] += 1
print("a[[0,0,1]] += 1 gives (repeat ignored):", c1)
c2 = np.zeros(4, dtype=int)
np.add.at(c2, [0, 0, 1], 1)
print("np.add.at(..., [0,0,1], 1) gives:", c2)
c3 = np.zeros(4, dtype=int)
np.add.at(c3, [0, 0, 1, 3], [5, 6, 7, 8])
print("add.at with per-index values:", c3)
mx = np.full(3, -np.inf)
np.maximum.at(mx, [0, 1, 0, 2, 1], [4.0, 1.0, 9.0, 2.0, 0.5])
print("maximum.at group maxima:", mx)
vals = np.arange(10)
print("np.add.reduceat(0..9, [0, 3, 7]) =", np.add.reduceat(vals, [0, 3, 7]), "(sums of 0:3, 3:7, 7:end)")
print("np.maximum.reduceat([1,5,2,8,3], [0, 2]) =", np.maximum.reduceat([1, 5, 2, 8, 3], [0, 2]))
print("# Gotcha: reduceat with a start index >= the next one returns just a[start], not an empty")
print("#   reduction. np.add.at is slower than bincount for plain counting, but more general.")
print()

a[[0,0,1]] += 1 gives (repeat ignored): [1 1 0 0]
np.add.at(..., [0,0,1], 1) gives: [2 1 0 0]
add.at with per-index values: [11  7  0  8]
maximum.at group maxima: [9. 1. 2.]
np.add.reduceat(0..9, [0, 3, 7]) = [ 3 18 24] (sums of 0:3, 3:7, 7:end)
np.maximum.reduceat([1,5,2,8,3], [0, 2]) = [5 8]
# Gotcha: reduceat with a start index >= the next one returns just a[start], not an empty
#   reduction. np.add.at is slower than bincount for plain counting, but more general.



## 06. frompyfunc and np.vectorize (and Why They Are Slow)

- np.vectorize(f): wraps a scalar Python function so it accepts arrays and broadcasts.
  It is a convenience for syntax, NOT a speed-up - it still calls f once per element.
- np.frompyfunc(f, nin, nout): real ufunc-like object but always returns object arrays.
- Real speed comes from rewriting with array ops (np.where, np.select, ufuncs).

In [7]:
def clamp_scale(v, lo):
    return v * 2 if v > lo else lo

vec = np.vectorize(clamp_scale)
data = np.array([1, 5, 3, 9])
print("np.vectorize result:", vec(data, 4), "(scalar lo broadcast)")
print("vectorize guesses output dtype from the FIRST result:", vec(data, 4).dtype.kind, "(i = integer)")
vec_f = np.vectorize(clamp_scale, otypes=[float])
print("with otypes=[float]:", vec_f(data, 4))
print("vectorized with np.where:", np.where(data > 4, data * 2, 4))
fp = np.frompyfunc(lambda s, n: s * n, 2, 1)
out = fp(np.array(["ab", "c"]), 2)
print("frompyfunc output dtype:", out.dtype, ", values:", out.tolist())
print("frompyfunc(...).astype(float) to get real numbers:", np.frompyfunc(lambda t: t + 0.5, 1, 1)(np.arange(3)).astype(float))
big = np.arange(20000)
print("same result both ways:", np.array_equal(np.vectorize(lambda t: t * 3 + 1)(big), big * 3 + 1))
print("# Gotcha: np.vectorize on an EMPTY input needs otypes= or it raises ValueError, and its")
print("#   first-result dtype guess can truncate: first output int -> later floats become ints.")
print("#   Rule: if you find yourself reaching for vectorize, look for where/select/ufuncs first.")
print()

np.vectorize result: [ 4 10  4 18] (scalar lo broadcast)
vectorize guesses output dtype from the FIRST result: i (i = integer)
with otypes=[float]: [ 4. 10.  4. 18.]
vectorized with np.where: [ 4 10  4 18]
frompyfunc output dtype: object , values: ['abab', 'cc']
frompyfunc(...).astype(float) to get real numbers: [0.5 1.5 2.5]
same result both ways: True
# Gotcha: np.vectorize on an EMPTY input needs otypes= or it raises ValueError, and its
#   first-result dtype guess can truncate: first output int -> later floats become ints.
#   Rule: if you find yourself reaching for vectorize, look for where/select/ufuncs first.



## 07. apply_along_axis and apply_over_axes

- apply_along_axis(f, axis, arr): call f on every 1-D slice along that axis.
  Still a Python loop underneath - use only when no built-in axis= version exists.
- apply_over_axes(f, a, axes): apply a reducing function over several axes in turn,
  keeping dimensions (so the result broadcasts back).

In [8]:
def spread(v):
    return v.max() - v.min()

grid = np.array([[3, 9, 1], [4, 4, 8], [7, 2, 5]])
print("apply_along_axis(spread, axis=1):", np.apply_along_axis(spread, 1, grid), "(per row)")
print("apply_along_axis(spread, axis=0):", np.apply_along_axis(spread, 0, grid), "(per column)")
print("built-in equivalent np.ptp(axis=1):", np.ptp(grid, axis=1))
print("apply_along_axis returning vectors (sorted desc per row):\n",
      np.apply_along_axis(lambda r: np.sort(r)[::-1], 1, grid))
cube = np.arange(24).reshape(2, 3, 4)
print("apply_over_axes(np.sum, cube, [0, 2]).shape:", np.apply_over_axes(np.sum, cube, [0, 2]).shape, "(kept as 1)")
print("same as cube.sum(axis=(0,2), keepdims=True):", np.array_equal(np.apply_over_axes(np.sum, cube, [0, 2]), cube.sum(axis=(0, 2), keepdims=True)))
print("# Gotcha: apply_along_axis is NOT vectorized - roughly as slow as a Python for-loop.")
print("#   Prefer arr.max(axis=...) etc. when the function is a standard reduction.")
print()

apply_along_axis(spread, axis=1): [8 4 5] (per row)
apply_along_axis(spread, axis=0): [4 7 7] (per column)
built-in equivalent np.ptp(axis=1): [8 4 5]
apply_along_axis returning vectors (sorted desc per row):
 [[9 3 1]
 [8 4 4]
 [7 5 2]]
apply_over_axes(np.sum, cube, [0, 2]).shape: (1, 3, 1) (kept as 1)
same as cube.sum(axis=(0,2), keepdims=True): True
# Gotcha: apply_along_axis is NOT vectorized - roughly as slow as a Python for-loop.
#   Prefer arr.max(axis=...) etc. when the function is a standard reduction.



## 08. einsum Step by Step

- einsum('subscripts', *arrays): write each array's axes as letters; a letter that
  appears in the inputs but NOT after '->' is summed over; letters after '->' stay.
- Repeated letter in ONE operand = diagonal. Output order is up to you.
- Read it as: for every combination of letters, multiply, then sum the dropped ones.

In [9]:
v1 = np.array([1, 2, 3])
v2 = np.array([4, 5, 6])
M = np.arange(6).reshape(2, 3)
N = np.arange(12).reshape(3, 4)
sq = np.array([[1, 2], [3, 4]])
print("1. sum of vector       'i->'     :", np.einsum("i->", v1))
print("2. elementwise product 'i,i->i'  :", np.einsum("i,i->i", v1, v2))
print("3. dot product         'i,i->'   :", np.einsum("i,i->", v1, v2), "(matches np.dot:", np.dot(v1, v2), ")")
print("4. outer product       'i,j->ij' :\n", np.einsum("i,j->ij", v1, v2))
print("5. transpose           'ij->ji'  :", np.einsum("ij->ji", M).shape)
print("6. column sums         'ij->j'   :", np.einsum("ij->j", M), ", row sums 'ij->i':", np.einsum("ij->i", M))
print("7. matrix product      'ij,jk->ik': equal to M @ N:", np.array_equal(np.einsum("ij,jk->ik", M, N), M @ N))
print("8. trace               'ii->'    :", np.einsum("ii->", sq), ", diagonal 'ii->i':", np.einsum("ii->i", sq))
print("9. matrix-vector       'ij,j->i' :", np.einsum("ij,j->i", M, v1), "(same as M @ v1:", M @ v1, ")")
print("10. row-wise dot       'ij,ij->i':", np.einsum("ij,ij->i", M, M), "(sum of squares per row)")
batch_a = np.arange(2 * 2 * 3).reshape(2, 2, 3)
batch_b = np.arange(2 * 3 * 2).reshape(2, 3, 2)
print("11. batched matmul 'bij,bjk->bik' matches np.matmul:", np.array_equal(np.einsum("bij,bjk->bik", batch_a, batch_b), np.matmul(batch_a, batch_b)))
print("12. all-pairs squared distance (3 pts, 2-D): 'id,jd->ij' on difference:")
pts = np.array([[0.0, 0.0], [3.0, 4.0], [6.0, 8.0]])
dots = np.einsum("id,jd->ij", pts, pts)
sqn = np.einsum("id,id->i", pts, pts)
print(sqn[:, None] + sqn[None, :] - 2 * dots)
print("13. implicit mode 'ij,jk' (no ->) sorts output letters alphabetically:", np.einsum("ij,jk", M, N).shape)
print("14. ellipsis '...ij->...ji' transposes the last two axes of anything:", np.einsum("...ij->...ji", batch_a).shape)
print("# Gotcha: a letter in the output that is not in any input is an error; a letter missing from")
print("#   the output is SUMMED. optimize=True may reorder multi-operand contractions for speed.")
print()

1. sum of vector       'i->'     : 6
2. elementwise product 'i,i->i'  : [ 4 10 18]
3. dot product         'i,i->'   : 32 (matches np.dot: 32 )
4. outer product       'i,j->ij' :
 [[ 4  5  6]
 [ 8 10 12]
 [12 15 18]]
5. transpose           'ij->ji'  : (3, 2)
6. column sums         'ij->j'   : [3 5 7] , row sums 'ij->i': [ 3 12]
7. matrix product      'ij,jk->ik': equal to M @ N: True
8. trace               'ii->'    : 5 , diagonal 'ii->i': [1 4]
9. matrix-vector       'ij,j->i' : [ 8 26] (same as M @ v1: [ 8 26] )
10. row-wise dot       'ij,ij->i': [ 5 50] (sum of squares per row)
11. batched matmul 'bij,bjk->bik' matches np.matmul: True
12. all-pairs squared distance (3 pts, 2-D): 'id,jd->ij' on difference:
[[  0.  25. 100.]
 [ 25.   0.  25.]
 [100.  25.   0.]]
13. implicit mode 'ij,jk' (no ->) sorts output letters alphabetically: (2, 4)
14. ellipsis '...ij->...ji' transposes the last two axes of anything: (2, 3, 2)
# Gotcha: a letter in the output that is not in any input is an error;

## 09. searchsorted

- np.searchsorted(sorted_arr, values, side='left'|'right'): binary search - returns the
  insertion index that keeps the array sorted. O(log n) per query.
- Also the tool for "which bin" / "nearest value" lookups.

In [10]:
s = np.array([10, 20, 20, 30, 40])
print("searchsorted(s, 20, 'left') =", np.searchsorted(s, 20, "left"), ", 'right' =", np.searchsorted(s, 20, "right"))
print("count of 20s =", np.searchsorted(s, 20, "right") - np.searchsorted(s, 20, "left"))
print("many queries at once:", np.searchsorted(s, [5, 25, 40, 99]))
print("exact membership test:", [bool(i < len(s) and s[i] == q) for i, q in zip(np.searchsorted(s, [20, 25]), [20, 25])])
q = np.array([12, 29, 38])
idx = np.clip(np.searchsorted(s, q), 1, len(s) - 1)
left, right = s[idx - 1], s[idx]
nearest = np.where(q - left <= right - q, left, right)
print("nearest value to [12,29,38]:", nearest)
print("with sorter= (unsorted data):", np.searchsorted(np.array([30, 10, 20]), 15, sorter=np.argsort([30, 10, 20])))
print("# Gotcha: input MUST be sorted ascending - otherwise results are silent garbage, no error.")
print()

searchsorted(s, 20, 'left') = 1 , 'right' = 3
count of 20s = 2
many queries at once: [0 3 4 5]
exact membership test: [True, False]
nearest value to [12,29,38]: [10 30 40]
with sorter= (unsorted data): 1
# Gotcha: input MUST be sorted ascending - otherwise results are silent garbage, no error.



## 10. digitize

- np.digitize(x, bins, right=False): bin number for each x. bins must be monotonic.
- Returns 0 for x below first edge and len(bins) for x above the last.
- Basically searchsorted with a different side convention (side='right' default).

In [11]:
ages = np.array([3, 17, 18, 40, 65, 90])
edges = np.array([18, 65])
labels = np.array(["child", "adult", "senior"])
b = np.digitize(ages, edges)
print("digitize(ages, [18, 65]) =", b, "->", labels[b].tolist())
print("right=True moves the boundary:", np.digitize(ages, edges, right=True))
print("same as searchsorted(side='right'):", np.array_equal(b, np.searchsorted(edges, ages, side="right")))
print("decreasing bins also work:", np.digitize([5, 15], [20, 10, 0]))
print("# Gotcha: digitize result 0 means 'below first edge' - indexes labels[b] must have len(bins)+1 items.")
print()

digitize(ages, [18, 65]) = [0 0 1 1 2 2] -> ['child', 'child', 'adult', 'adult', 'senior', 'senior']
right=True moves the boundary: [0 0 0 1 1 2]
same as searchsorted(side='right'): True
decreasing bins also work: [2 1]
# Gotcha: digitize result 0 means 'below first edge' - indexes labels[b] must have len(bins)+1 items.



## 11. np.unique with return_*

- np.unique(a): sorted unique values (flattens the input unless axis= is given).
- return_index: position of first occurrence. return_inverse: rebuilds the original
  from unique values. return_counts: how many times each appears.
- np.unique(rows, axis=0) finds unique ROWS.

In [12]:
items = np.array([3, 1, 2, 3, 1, 3])
u, first, inv, cnt = np.unique(items, return_index=True, return_inverse=True, return_counts=True)
print("unique:", u, ", first idx:", first, ", counts:", cnt)
print("inverse:", inv, "-> u[inv] rebuilds:", u[inv])
print("most common value:", u[np.argmax(cnt)])
rows = np.array([[1, 2], [3, 4], [1, 2], [5, 6]])
print("unique rows (axis=0):\n", np.unique(rows, axis=0))
print("words unique:", np.unique(np.array(["pear", "apple", "pear", "fig"])).tolist())
print("keep ORIGINAL order of first appearance:", items[np.sort(first)])
print("duplicate flags:", ~np.isin(np.arange(len(items)), first))
print("np.unique(2-D) flattens:", np.unique(np.array([[2, 1], [2, 3]])))
print("# Gotcha: output is SORTED, not in appearance order. Use return_index to recover first-seen order.")
print("#   For 2-D input without axis=, the result (and inverse) are flattened.")
print()

unique: [1 2 3] , first idx: [1 2 0] , counts: [2 1 3]
inverse: [2 0 1 2 0 2] -> u[inv] rebuilds: [3 1 2 3 1 3]
most common value: 3
unique rows (axis=0):
 [[1 2]
 [3 4]
 [5 6]]
words unique: ['apple', 'fig', 'pear']
keep ORIGINAL order of first appearance: [3 1 2]
duplicate flags: [False False False  True  True  True]
np.unique(2-D) flattens: [1 2 3]
# Gotcha: output is SORTED, not in appearance order. Use return_index to recover first-seen order.
#   For 2-D input without axis=, the result (and inverse) are flattened.



## 12. bincount

- np.bincount(x): counts of each non-negative INT value 0..max. Very fast.
- weights= turns counts into per-bin sums. minlength= pads the output.
- Compare with np.unique counts: bincount is faster but only for small non-negative ints.

In [13]:
labels_arr = np.array([0, 2, 2, 1, 2, 0])
print("bincount:", np.bincount(labels_arr))
print("minlength=5:", np.bincount(labels_arr, minlength=5))
w = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
sums = np.bincount(labels_arr, weights=w)
print("group sums by label (weights):", sums)
print("group means:", sums / np.bincount(labels_arr))
print("confusion-style 2-D count via ravel_multi_index:",
      np.bincount(np.ravel_multi_index(([0, 1, 1, 1], [0, 1, 1, 0]), (2, 2)), minlength=4).reshape(2, 2).tolist())
try:
    np.bincount(np.array([-1, 2]))
except ValueError as e:
    print("negative value ->", type(e).__name__)
print("# Gotcha: bincount output length is max+1 - a single huge value like 10**9 allocates a")
print("#   giant array. Use np.unique(return_inverse=True) first to compress sparse labels.")
print()

bincount: [2 1 3]
minlength=5: [2 1 3 0 0]
group sums by label (weights): [ 7.  4. 10.]
group means: [3.5        4.         3.33333333]
confusion-style 2-D count via ravel_multi_index: [[1, 0], [1, 2]]
negative value -> ValueError
# Gotcha: bincount output length is max+1 - a single huge value like 10**9 allocates a
#   giant array. Use np.unique(return_inverse=True) first to compress sparse labels.



## 13. histogram and histogram2d

- np.histogram(x, bins, range) -> (counts, edges). bins can be an int or explicit edges.
- All bins are half-open [a, b) EXCEPT the last, which is closed [a, b].
- density=True makes the area (not the height) sum to 1.
- histogram2d(x, y, bins) -> counts grid, x edges, y edges.

In [14]:
h_data = np.array([1, 2, 2, 3, 3, 3, 4, 4, 4, 4, 10])
counts, edges = np.histogram(h_data, bins=[0, 2, 4, 6, 10])
print("counts:", counts, ", edges:", edges)
print("value 10 (== last edge) is included in the last bin:", counts[-1])
print("bins=3 auto edges:", np.histogram(h_data, bins=3)[1])
dens, e2 = np.histogram(h_data, bins=[0, 2, 4, 6, 10], density=True)
print("density sums with widths to 1:", np.sum(dens * np.diff(e2)))
print("histogram with range=(0, 5):", np.histogram(h_data, bins=5, range=(0, 5))[0], "(10 is outside, dropped)")
hx = np.array([0.5, 1.5, 1.5, 2.5])
hy = np.array([0.5, 0.5, 1.5, 1.5])
H, xe, ye = np.histogram2d(hx, hy, bins=[[0, 1, 2, 3], [0, 1, 2]])
print("histogram2d counts (rows=x bins, cols=y bins):\n", H)
print("np.histogram_bin_edges(h_data, bins='auto') length:", len(np.histogram_bin_edges(h_data, bins="auto")))
print("# Gotcha: histogram2d rows are the X bins and columns are the Y bins - the opposite of how")
print("#   an image/plot is drawn, so transpose before imshow.")
print()

counts: [1 5 4 1] , edges: [ 0  2  4  6 10]
value 10 (== last edge) is included in the last bin: 1
bins=3 auto edges: [ 1.  4.  7. 10.]
density sums with widths to 1: 1.0
histogram with range=(0, 5): [0 1 2 3 4] (10 is outside, dropped)
histogram2d counts (rows=x bins, cols=y bins):
 [[1. 0.]
 [1. 1.]
 [0. 1.]]
np.histogram_bin_edges(h_data, bins='auto') length: 8
# Gotcha: histogram2d rows are the X bins and columns are the Y bins - the opposite of how
#   an image/plot is drawn, so transpose before imshow.



## 14. Set Operations

- union1d, intersect1d, setdiff1d, setxor1d: return SORTED unique results.
- np.isin(a, b): boolean mask of "is each element of a in b" - keeps a's shape.
  (in1d is the older flat version; isin replaces it.)
- intersect1d(return_indices=True) gives where the common values sit.

In [15]:
p1 = np.array([5, 1, 3, 3, 7])
p2 = np.array([3, 7, 9, 9])
print("union1d:", np.union1d(p1, p2))
print("intersect1d:", np.intersect1d(p1, p2))
print("setdiff1d(p1, p2):", np.setdiff1d(p1, p2))
print("setxor1d:", np.setxor1d(p1, p2))
print("isin(p1, p2):", np.isin(p1, p2), ", invert=True:", np.isin(p1, p2, invert=True))
print("filter p1 by membership:", p1[np.isin(p1, p2)])
common, i1, i2 = np.intersect1d(p1, p2, return_indices=True)
print("intersect indices:", common, i1, i2)
print("isin keeps 2-D shape:", np.isin(np.array([[1, 5], [9, 3]]), p1).tolist())
print("# Gotcha: results are sorted and de-duplicated - not the order of the input. isin is the")
print("#   one that preserves order and duplicates in the first argument.")
print()

union1d: [1 3 5 7 9]
intersect1d: [3 7]
setdiff1d(p1, p2): [1 5]
setxor1d: [1 5 9]
isin(p1, p2): [False False  True  True  True] , invert=True: [ True  True False False False]
filter p1 by membership: [3 3 7]
intersect indices: [3 7] [2 4] [0 1]
isin keeps 2-D shape: [[True, True], [False, True]]
# Gotcha: results are sorted and de-duplicated - not the order of the input. isin is the
#   one that preserves order and duplicates in the first argument.



## 15. Sorting and Sort Kinds

- np.sort(a) returns a sorted COPY; a.sort() sorts in place and returns None.
- np.argsort gives the indexes that would sort. axis=-1 by default (sorts each row).
- kind: 'quicksort' (default, introsort, not stable), 'mergesort'/'stable' (stable),
  'heapsort'. Use 'stable' when ties must keep input order or for multi-pass sorting.
- Descending: sort then [::-1], or negate numbers. a[::-1] is a view.

In [16]:
raw = np.array([3, 1, 2, 1, 3])
print("np.sort:", np.sort(raw), ", original untouched:", raw)
tmp = raw.copy()
print("in-place tmp.sort() returns:", tmp.sort(), ", tmp is now", tmp)
print("argsort kind='stable' (ties keep original order):", np.argsort(raw, kind="stable"))
print("descending stable by negation:", np.argsort(-raw, kind="stable"))
two = np.array([[3, 1, 2], [9, 8, 7]])
print("sort axis=1 (default):", np.sort(two, axis=1).tolist(), ", axis=0:", np.sort(two, axis=0).tolist())
print("sort axis=None flattens:", np.sort(two, axis=None))
recs = np.array([("b", 2), ("a", 2), ("c", 1)], dtype=[("name", "U1"), ("n", "i4")])
print("sort structured by field 'n' then name:", np.sort(recs, order=["n", "name"]).tolist())
print("sorted check:", bool(np.all(np.diff(np.sort(raw)) >= 0)))
print("strings sort lexicographically:", np.sort(np.array(["b10", "b2", "a"])).tolist())
print("# Gotcha: the default sort is NOT stable. Two-pass sorts (secondary key first, then primary)")
print("#   only work with kind='stable'. Also, NaNs sort to the END.")
print("nan sort:", np.sort(np.array([2.0, np.nan, 1.0])))
print()

np.sort: [1 1 2 3 3] , original untouched: [3 1 2 1 3]
in-place tmp.sort() returns: None , tmp is now [1 1 2 3 3]
argsort kind='stable' (ties keep original order): [1 3 2 0 4]
descending stable by negation: [0 4 2 1 3]
sort axis=1 (default): [[1, 2, 3], [7, 8, 9]] , axis=0: [[3, 1, 2], [9, 8, 7]]
sort axis=None flattens: [1 2 3 7 8 9]
sort structured by field 'n' then name: [('c', 1), ('a', 2), ('b', 2)]
sorted check: True
strings sort lexicographically: ['a', 'b10', 'b2']
# Gotcha: the default sort is NOT stable. Two-pass sorts (secondary key first, then primary)
#   only work with kind='stable'. Also, NaNs sort to the END.
nan sort: [ 1.  2. nan]



## 16. partition and argpartition

- np.partition(a, k): puts the k-th smallest in sorted position; smaller values before,
  larger after, but neither side is sorted. O(n) instead of O(n log n).
- Perfect for "top-k" and median-like questions on large arrays.

In [17]:
pa = np.array([9, 4, 7, 1, 8, 3, 6])
print("sorted for reference:", np.sort(pa))
part = np.partition(pa, 2)
print("partition(k=2):", part, "-> element [2] =", part[2], "(3rd smallest, in its final place)")
print("3 smallest (any order):", np.sort(np.partition(pa, 2)[:3]))
k = 3
topk_idx = np.argpartition(pa, -k)[-k:]
print("indexes of top-3:", np.sort(topk_idx), "values:", np.sort(pa[topk_idx])[::-1])
print("median via partition:", np.partition(pa, len(pa) // 2)[len(pa) // 2], "vs np.median:", np.median(pa))
print("multiple kth: partition(pa, [1, 4])[[1, 4]] =", np.partition(pa, [1, 4])[[1, 4]])
mat = np.array([[5, 2, 9], [1, 8, 3]])
print("argpartition per row (smallest 1):", np.argpartition(mat, 0, axis=1)[:, 0])
print("# Gotcha: only position k is guaranteed. The rest of the 'small half' is in arbitrary")
print("#   order - sort that slice yourself if you need top-k ordered.")
print()

sorted for reference: [1 3 4 6 7 8 9]
partition(k=2): [1 3 4 9 8 7 6] -> element [2] = 4 (3rd smallest, in its final place)
3 smallest (any order): [1 3 4]
indexes of top-3: [0 2 4] values: [9 8 7]
median via partition: 6 vs np.median: 6.0
multiple kth: partition(pa, [1, 4])[[1, 4]] = [3 7]
argpartition per row (smallest 1): [1 0]
# Gotcha: only position k is guaranteed. The rest of the 'small half' is in arbitrary
#   order - sort that slice yourself if you need top-k ordered.



## 17. lexsort

- np.lexsort((secondary, primary)): sort by several keys. The LAST key is the PRIMARY one.
- Returns indexes (like argsort). Stable. Works on columns of separate arrays.

In [18]:
last = np.array(["Lee", "Kim", "Lee", "Kim"])
first_n = np.array(["Ann", "Zed", "Bob", "Amy"])
order = np.lexsort((first_n, last))
print("lexsort by last name then first name:", list(zip(last[order], first_n[order])))
score = np.array([90, 85, 90, 85])
order2 = np.lexsort((first_n, -score))
print("score desc then first name:", list(zip(score[order2].tolist(), first_n[order2])))
table = np.array([[2, 1], [1, 9], [2, 0], [1, 3]])
print("sort rows by col0 then col1:\n", table[np.lexsort((table[:, 1], table[:, 0]))])
print("# Gotcha: key order is REVERSED from what people expect (last key sorts first).")
print("#   Negate numeric keys for descending; strings cannot be negated.")
print()

lexsort by last name then first name: [('Kim', 'Amy'), ('Kim', 'Zed'), ('Lee', 'Ann'), ('Lee', 'Bob')]
score desc then first name: [(90, 'Ann'), (90, 'Bob'), (85, 'Amy'), (85, 'Zed')]
sort rows by col0 then col1:
 [[1 3]
 [1 9]
 [2 0]
 [2 1]]
# Gotcha: key order is REVERSED from what people expect (last key sorts first).
#   Negate numeric keys for descending; strings cannot be negated.



## 18. NaN-Aware Functions

- nansum, nanmean, nanstd, nanvar, nanmin, nanmax, nanargmin, nanargmax, nanmedian,
  nanpercentile, nanprod, nancumsum, nancumprod ignore nans.
- They treat all-nan slices differently (warn, return nan) - decide how to handle.

In [19]:
nv = np.array([[1.0, np.nan, 3.0], [np.nan, np.nan, np.nan]])
print("sum:", nv.sum(axis=1), "| nansum:", np.nansum(nv, axis=1), "(all-nan row sums to 0.0)")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    print("nanmean per row:", np.nanmean(nv, axis=1), "(all-nan -> nan, with a RuntimeWarning)")
print("nanmax:", np.nanmax(nv[0]), ", nanargmax:", np.nanargmax(nv[0]))
print("nanstd of [1,nan,3]:", np.nanstd(nv[0]))
print("nanmedian:", np.nanmedian(np.array([1.0, np.nan, 5.0, 9.0])))
print("nancumsum:", np.nancumsum(np.array([1.0, np.nan, 2.0])))
print("count valid per column:", np.count_nonzero(~np.isnan(nv), axis=0))
print("mean with fill:", np.where(np.isnan(nv), np.nanmean(nv[0]), nv)[0])
print("# Gotcha: np.max / np.mean propagate nan, np.nanmax / np.nanmean skip it. nanargmax raises on an")
print("#   all-nan slice. Integer arrays cannot hold nan, so nan-functions are for floats.")
print()

sum: [nan nan] | nansum: [4. 0.] (all-nan row sums to 0.0)
nanmean per row: [ 2. nan] (all-nan -> nan, with a RuntimeWarning)
nanmax: 3.0 , nanargmax: 2
nanstd of [1,nan,3]: 1.0
nanmedian: 5.0
nancumsum: [1. 1. 3.]
count valid per column: [1 0 1]
mean with fill: [1. 2. 3.]
# Gotcha: np.max / np.mean propagate nan, np.nanmax / np.nanmean skip it. nanargmax raises on an
#   all-nan slice. Integer arrays cannot hold nan, so nan-functions are for floats.



## 19. Cumulative Operations

- cumsum, cumprod, np.maximum.accumulate (running max), np.minimum.accumulate.
- axis= picks direction. The output has the same shape as the input.
- cumsum - shifted = windowed sums (see section 20 for diff).

In [20]:
cv = np.array([2, 1, 4, 3])
print("cumsum:", np.cumsum(cv), ", cumprod:", np.cumprod(cv))
cm = np.arange(6).reshape(2, 3)
print("cumsum axis=0:", np.cumsum(cm, axis=0).tolist(), ", axis=1:", np.cumsum(cm, axis=1).tolist())
print("cumsum of 2-D with axis=None flattens:", np.cumsum(cm))
print("running max:", np.maximum.accumulate(cv), ", running min:", np.minimum.accumulate(cv))
print("fraction of total so far:", np.cumsum(cv) / cv.sum())
cs = np.concatenate(([0], np.cumsum(cv)))
print("sum of cv[1:3] via prefix sums:", cs[3] - cs[1], "(same as", cv[1:3].sum(), ")")
print("cumsum on a bool counts Trues so far:", np.cumsum(np.array([True, False, True, True])))
print("cumsum of int8 widens the accumulator:", np.cumsum(np.array([100, 100], dtype=np.int8)).tolist(), "(no int8 wrap)")
print("cumprod 1..25 last value in int64:", int(np.cumprod(np.arange(1, 26, dtype=np.int64))[-1]), "(wrong; true 25! is much larger)")
print("# Gotcha: cumprod grows fast - int64 overflow wraps silently, as the line above shows.")
print()

cumsum: [ 2  3  7 10] , cumprod: [ 2  2  8 24]
cumsum axis=0: [[0, 1, 2], [3, 5, 7]] , axis=1: [[0, 1, 3], [3, 7, 12]]
cumsum of 2-D with axis=None flattens: [ 0  1  3  6 10 15]
running max: [2 2 4 4] , running min: [2 1 1 1]
fraction of total so far: [0.2 0.3 0.7 1. ]
sum of cv[1:3] via prefix sums: 5 (same as 5 )
cumsum on a bool counts Trues so far: [1 1 2 3]
cumsum of int8 widens the accumulator: [100, 200] (no int8 wrap)
cumprod 1..25 last value in int64: 7034535277573963776 (wrong; true 25! is much larger)
# Gotcha: cumprod grows fast - int64 overflow wraps silently, as the line above shows.



## 20. diff, gradient, convolve, correlate

- np.diff(a, n, axis): differences of neighbours (length shrinks by 1 per pass).
- np.gradient(a): numeric derivative; same length as input (central differences).
- np.convolve(a, v, mode): sliding weighted sum with the kernel FLIPPED.
- np.correlate(a, v, mode): same without flipping. 'valid', 'same', 'full' modes.

In [21]:
sq_t = np.array([0, 1, 4, 9, 16, 25])
print("diff:", np.diff(sq_t), ", second diff (n=2):", np.diff(sq_t, n=2))
print("diff on 2-D axis=1:", np.diff(np.array([[1, 4, 9], [2, 2, 2]]), axis=1).tolist())
print("diff with prepend to keep length:", np.diff(sq_t, prepend=0))
print("gradient (unit spacing):", np.gradient(sq_t.astype(float)))
print("gradient with x coords:", np.gradient(sq_t.astype(float), np.array([0, 2, 4, 6, 8, 10.0])))
sig = np.array([1, 2, 3, 4, 5], dtype=float)
box = np.ones(3) / 3
print("convolve 'valid' (moving avg):", np.convolve(sig, box, mode="valid"))
print("convolve 'same':", np.convolve(sig, box, mode="same"))
print("convolve 'full' length:", len(np.convolve(sig, box, mode="full")))
print("kernel flip matters: convolve with [1,0,-1]:", np.convolve(sig, [1, 0, -1], "valid"),
      "| correlate:", np.correlate(sig, [1, 0, -1], "valid"))
print("polynomial multiply via convolve (1+x)(1+x):", np.convolve([1, 1], [1, 1]))
sig2 = np.array([0, 1, 0.5])
print("correlate full (signal vs itself):", np.correlate(sig2, sig2, "full"))
print("# Gotcha: convolve is 1-D only (use scipy.signal for 2-D). Edges in 'same' mode are")
print("#   zero-padded, so the first/last averages are pulled toward 0.")
print()

diff: [1 3 5 7 9] , second diff (n=2): [2 2 2 2]
diff on 2-D axis=1: [[3, 5], [0, 0]]
diff with prepend to keep length: [0 1 3 5 7 9]
gradient (unit spacing): [1. 2. 4. 6. 8. 9.]
gradient with x coords: [0.5 1.  2.  3.  4.  4.5]
convolve 'valid' (moving avg): [2. 3. 4.]
convolve 'same': [1. 2. 3. 4. 3.]
convolve 'full' length: 7
kernel flip matters: convolve with [1,0,-1]: [2. 2. 2.] | correlate: [-2. -2. -2.]
polynomial multiply via convolve (1+x)(1+x): [1 2 1]
correlate full (signal vs itself): [0.   0.5  1.25 0.5  0.  ]
# Gotcha: convolve is 1-D only (use scipy.signal for 2-D). Edges in 'same' mode are
#   zero-padded, so the first/last averages are pulled toward 0.



## 21. clip, round, floor Variants

- np.clip(a, lo, hi): limit values. Either bound may be None.
- np.round / around: round half to EVEN. decimals=-1 rounds to tens.
- floor (toward -inf), ceil (toward +inf), trunc (toward 0), rint (nearest, half-even).
- floor_divide (//) and mod (%) use floor semantics; np.fmod / np.remainder differ for negatives.

In [22]:
r = np.array([-2.5, -1.5, -0.5, 0.5, 1.5, 2.5, 2.7, -2.7])
print("values   :", r)
print("round    :", np.round(r))
print("rint     :", np.rint(r), "(same as round)")
print("floor    :", np.floor(r))
print("ceil     :", np.ceil(r))
print("trunc    :", np.trunc(r))
print("astype(int) truncates too:", r.astype(int))
print("round decimals=1:", np.round(np.array([1.25, 1.35, 2.675]), 1), "(float storage error + half-even)")
print("round decimals=-1:", np.round(np.array([14, 15, 25, 36]), -1))
print("clip(0, 1):", np.clip(np.array([-0.5, 0.3, 1.7]), 0, 1), ", lower only:", np.clip(np.array([-3, 2, 9]), 0, None))
print("-7 // 2 =", np.floor_divide(-7, 2), ", -7 % 2 =", np.mod(-7, 2), "(floor-based)")
print("np.fmod(-7, 2) =", np.fmod(-7, 2), "(sign follows dividend, like C)")
print("np.divmod(-7, 2):", tuple(int(t) for t in np.divmod(-7, 2)))
print("np.sign:", np.sign(np.array([-3, 0, 5])), ", np.abs:", np.abs(np.array([-3, 0, 5])))
print("# Gotcha: np.round(2.5) is 2.0 and round(3.5) is 4.0 (half to even), not school rounding.")
print("#   For half-up use np.floor(x + 0.5). Also astype(int) truncates toward zero, not floor.")
print()

values   : [-2.5 -1.5 -0.5  0.5  1.5  2.5  2.7 -2.7]
round    : [-2. -2. -0.  0.  2.  2.  3. -3.]
rint     : [-2. -2. -0.  0.  2.  2.  3. -3.] (same as round)
floor    : [-3. -2. -1.  0.  1.  2.  2. -3.]
ceil     : [-2. -1. -0.  1.  2.  3.  3. -2.]
trunc    : [-2. -1. -0.  0.  1.  2.  2. -2.]
astype(int) truncates too: [-2 -1  0  0  1  2  2 -2]
round decimals=1: [1.2 1.4 2.7] (float storage error + half-even)
round decimals=-1: [10 20 20 40]
clip(0, 1): [0.  0.3 1. ] , lower only: [0 2 9]
-7 // 2 = -4 , -7 % 2 = 1 (floor-based)
np.fmod(-7, 2) = -1 (sign follows dividend, like C)
np.divmod(-7, 2): (-4, 1)
np.sign: [-1  0  1] , np.abs: [3 0 5]
# Gotcha: np.round(2.5) is 2.0 and round(3.5) is 4.0 (half to even), not school rounding.
#   For half-up use np.floor(x + 0.5). Also astype(int) truncates toward zero, not floor.



## 22. Cross-References

- 02-Indexing-And-Slicing: fancy and boolean indexing (take/put build on it).
- 05-Statistical-And-Math-Functions: sum/mean/percentile (here: nan-aware, cumulative).
- 06-Linear-Algebra: matmul/dot (here: einsum as the general form).
- 08-Performance-And-Vectorization: why loops lose (here: vectorize/apply_along_axis are loops).
- 10-Advanced-NumPy: structured arrays, np.select, meshgrid.
- 11-Strides-And-Memory-Layout: views vs copies, which explains put/take behaviour.
- 13-NumPy-Recipes-And-Real-Patterns: these tools combined in real tasks.

In [23]:
print("Rule of thumb: pick the narrowest tool - take/put for axis picks, bincount/unique for counting,")
print("partition for top-k, searchsorted for lookups, einsum when axes need names.")

Rule of thumb: pick the narrowest tool - take/put for axis picks, bincount/unique for counting,
partition for top-k, searchsorted for lookups, einsum when axes need names.
